---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-08: Running Open-Source AI Models Locally with Ollama</h1>

# Learning agenda of this notebook

1. **Local LLM Runtime Frameworks:** What they are, and how the main ones differ
2. **Ollama:** What it is, and installing it (architecture, install, first check)
3. **Using Ollama through the GUI:** Free cloud-hosted models first, then a small local model
4. **Using Ollama through the CLI:** The same two kinds of model from your terminal, plus the raw REST API
5. **Agent harnesses on Ollama:** Claude Code, Codex CLI, OpenCode, Aider, Goose, and pointing one at Ollama
6. **Programmatic access:** Ollama's native Python library, then OpenAI's `Chat Completions` and `Responses` APIs
7. **Hello-World Examples:** QA, classification, generation, translation, summarization, NER
8. **GGUF Models and llama.cpp:** Building a custom Ollama model from a GGUF file, then running the same file with llama.cpp
9. **Beyond llama.cpp:** vLLM and AirLLM
10. **Build Your Own Inference Server:** An engine with PyTorch, then an OpenAI-compatible server with FastAPI + uvicorn

# <span style='background :lightgreen' >Recap: Ways to Access Open Source LLMs</span>
### (i) Access Open-Source Models via Cloud-Based Providers (Driving a fully automatic car — everything managed for you)

### (ii) Run Open-Source Models locally using runtimes (Driving an automatic car — local but simple, no gears or engineering)

### (iii) Use Open-Source Models via Hugging Face `pipeline()` API (Driving a manual car — you see more of the mechanics, but still a car someone else built)

### (iv) Load and run models directly from Hugging Face Hub using `AutoModel/AutoTokenizer` (Opening the hood and adjusting or replacing engine components)


### (v) Fine-Tune LLMs using full fine-tuning or PEFT methods (LoRA / QLoRA / adapters) (Upgrading and re-calibrating the engine to suit your driving style)

### (vi) Build and train an AI Model from scratch using PyTorch / TensorFlow (Designing and building the entire car from raw parts — full control, full responsibility)


# <span style='background :lightgreen' >1. Local LLM Runtime Frameworks</span>

## Why is Local Inference Significant?
1. **Privacy & Security:** Data stays within your own system, so sensitive information is not sent to third-party cloud servers. This is especially important in healthcare, finance, and government sectors. Keeping data local lowers exposure to external threats and minimizes the risk of data breaches.
2. **Cost Efficiency:** Reduces cloud-related expenses such as API usage fees, bandwidth, and compute costs.
3. **Offline Operation:** Models continue to work without an internet connection. Useful in environments with limited or unreliable connectivity (e.g., remote areas, edge devices).
4. **Low Latency:** Running models locally reduces network delay, resulting in faster responses. This is critical for real-time applications like chatbots and interactive systems.
5. **Customization & Control:** You have full control over the model, data, updates, and fine-tuning without platform restrictions.
6. **Regulatory Compliance:** Helps organizations follow data privacy laws by keeping sensitive data inside their own systems. This supports compliance with regulations such as GDPR (European Union data protection law) and CCPA (California’s data privacy law), which require careful handling and protection of personal data.

## How the pieces fit together

Before comparing tools, it helps to see that they are **not competing products** — they sit at different layers of the same stack:

```
   You  ──▶  Experience layer  ──▶  Inference engine  ──▶  Your hardware
             (Ollama, LM Studio,     (llama.cpp, MLX)       (CPU / GPU /
              GPT4All)                                       Apple Silicon)

   Production serving:  vLLM, SGLang, TensorRT-LLM  ──▶  their own engines  ──▶  GPU cluster
   Python libraries:    HF Transformers, AirLLM     ──▶  PyTorch            ──▶  CPU / GPU
```

- **Engines** do the actual maths: `llama.cpp` (portable C/C++, runs almost anywhere) and `MLX` (Apple's framework, fastest on M-series chips).
- **Experience layers** wrap an engine and add model downloading, versioning, a chat UI, and an API: Ollama, LM Studio, GPT4All.
- **Serving systems** are built for many simultaneous users rather than one person: vLLM, SGLang, TensorRT-LLM.
- **Python libraries** load a model inside your own program: Hugging Face Transformers (Lec-12, Lec-21), and AirLLM, which fits a huge model into small memory.

> This is why you will read that "Ollama is slightly slower than llama.cpp" — Ollama *runs* llama.cpp underneath and adds convenience on top. On Apple Silicon, Ollama 0.19 (March 2026) added an **MLX** backend as a preview, but only for Macs with **more than 32 GB** of unified memory; Macs with less memory keep using llama.cpp on the Mac's GPU (Metal).

## Comparing the Main Runtimes

| Runtime | Layer | Interface | Best at | Watch out for |
|---|---|---|---|---|
| **[Ollama](https://ollama.com/)** | Experience layer (over llama.cpp / MLX) | CLI + desktop GUI + REST API | One-developer prototyping on any OS; OpenAI-compatible API with almost no setup | Thin wrapper overhead; not built for many concurrent users |
| **[LM Studio](https://lmstudio.ai/)** | Experience layer (over llama.cpp / MLX) | GUI-first (Windows/macOS/Linux) | Browsing, downloading and chatting with models with **no coding at all** | GUI-centric; less natural to script |
| **[GPT4All](https://nomic.ai/gpt4all)** | Experience layer | Desktop app + SDKs (Python, Node.js, C++) | **LocalDocs** — point it at a folder of PDFs and chat with them fully offline | Entry-level serving features only |
| **[llama.cpp](https://github.com/ggml-org/llama.cpp)** | **Engine** | CLI + lightweight HTTP server | Edge devices, odd hardware, aggressive quantization, maximum control | You manage GGUF files and flags yourself; no model manager |
| **[vLLM](https://github.com/vllm-project/vllm)** | **Serving system** | OpenAI-compatible HTTP server | Production, multi-user, high concurrency: about **19x** Ollama's peak throughput in [Red Hat's test](https://developers.redhat.com/articles/2025/08/08/ollama-vs-vllm-deep-dive-performance-benchmarking) (793 vs 41 tokens/s on one A100), thanks to PagedAttention + continuous batching | GPU-oriented; heavier to deploy; overkill for one user |
| **[SGLang](https://github.com/sgl-project/sglang)** | **Serving system** | OpenAI-compatible HTTP server | Many requests that share the same prompt prefix (agents, RAG, multi-turn chat): RadixAttention reuses their KV cache | GPU-oriented; smaller ecosystem than vLLM |
| **[TensorRT-LLM](https://github.com/NVIDIA/TensorRT-LLM)** | **Serving system** | Python API + `trtllm-serve` | Lowest latency on NVIDIA GPUs | NVIDIA only; the heaviest setup of all |
| **[MLX / mlx-lm](https://github.com/ml-explore/mlx-lm)** | **Engine** | Python library + CLI | The fastest option on Apple Silicon Macs | Apple Silicon only |
| **[HF Transformers](https://github.com/huggingface/transformers)** | **Python library** | `pipeline()`, `AutoModel` | Every Hub model on release day; research and fine-tuning | Slow and memory-hungry; no batching for many users |
| **[AirLLM](https://github.com/lyogavin/airllm)** | **Python library** | Python | Models far bigger than your RAM or GPU memory, by loading one layer at a time | Very slow (seconds per token); needs disk space for a per-layer copy of the model |
| **Build your own** (PyTorch + FastAPI + uvicorn) | **Engine and server, written by you** | OpenAI-compatible HTTP API | Learning what an inference server really does (section 10) | One user at a time; no quantization, batching or model manager |

> **Also worth knowing: [MLX](https://github.com/ml-explore/mlx)** is Apple's array framework for Apple Silicon. LM Studio can use it for you on M-series Macs, Ollama uses it on Macs with more than 32 GB, and `mlx-lm` runs it directly from Python. It is the reason local inference is fast on modern MacBooks.

### Which should you pick?

| If you want to... | Use |
|---|---|
| Prototype on your own laptop, any OS | **Ollama** |
| Click around a model browser, never touch a terminal | **LM Studio** |
| Chat with your own PDFs, entirely offline | **GPT4All** (LocalDocs) |
| Deploy to embedded or unusual hardware | **llama.cpp** directly |
| Serve many users in production on GPUs | **vLLM** |
| Serve agents or RAG with long shared prompts at scale | **SGLang** |
| Get the lowest latency out of NVIDIA GPUs in production | **TensorRT-LLM** |
| Squeeze maximum speed out of an M-series Mac | **MLX** (via LM Studio or `mlx-lm`) |
| Do research or fine-tuning, or look inside a model | **HF Transformers** |
| Run a model bigger than your memory, when speed does not matter | **AirLLM** |
| Understand how an inference server works | **Build your own** (section 9d) |

**This notebook uses Ollama**, because it is the shortest path from "nothing installed" to "a model answering my Python code" — and because its OpenAI-compatible API means everything you learned in the previous lectures transfers unchanged.

- [PagedAttention paper (vLLM)](https://arxiv.org/pdf/2309.06180)
- [Ollama vs vLLM benchmark (Red Hat, 2025)](https://developers.redhat.com/articles/2025/08/08/ollama-vs-vllm-deep-dive-performance-benchmarking)

> **Older tutorials often recommend Hugging Face TGI** (Text Generation Inference). It is now in maintenance mode, and Hugging Face points users to vLLM, SGLang, llama.cpp or MLX instead.

# <span style='background :lightgreen' >2. Ollama: What It Is, and Installing It</span>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">Ollama is an open-source platform that makes it easier to work with open-source AI models, providing features like model downloading, running it on your local box, and interacting through CLI or via APIs</div></h3>


## a. Architecture of Ollama
- **Server Component:**
    - When you install and run Ollama, it launches a local server that listens on http://localhost:11434
    - This server is responsible for managing models (loading, unloading, switching) and handling inference requests.
    - The API it exposes follows a REST architecture, making it accessible through HTTP requests.
    - So whatever LLM you choose to run using  ollama, your request at the end of the day will be passed to the ollama server, which runs on your localhost at port 11434.
- **Client Interaction:**
    - Applications can interact with the Ollama server in three  ways:
        - Using the official Ollama CLI, which directly communicate with the local server. 
        - Using Ollama-native client like `ollama.chat()` in Python or JavaScript.
        - Using the OpenAI client library and using `client.chat.completions.create()` or `client.responses.create()`, since Ollama exposes endpoint (http://localhost:11434/v1) that mimic the OpenAI API format.
    - Clients can issue requests for text generation, chat-style completions, streaming responses, and model management.
    - The Ollama server then handles the request, executes inference on your local hardware, and returns the response to the client.
- **Networked Environment:**
    - Although Ollama defaults to localhost, the server can be run on a different machine in the same network.
    - In that setup, clients simply target the remote machine’s IP and port 11434.
    - This allows you to centralize inference on a powerful machine (GPU server) while keeping clients lightweight.
    - So Ollama can run on a private-IP machine inside a LAN and be accessed locally via CLI/GUI or remotely via HTTP using Python clients (including the OpenAI-compatible API), and it can also be deployed on a public-IP machine for internet access, but doing so safely requires a reverse proxy, authentication, and TLS since Ollama itself provides no built-in security.


## b. Download and Install Ollama
- There are different ways to download and install **ollama**:
    1. Download Ollama for your Mac, Linux or Windows machine by visiting: https://ollama.com/download
    2. Download docker image of ollama available on dockerhub and run it inside a container
```
Linux/MacOS: curl -fsSL https://ollama.com/install.sh | sh
Windows: irm https://ollama.com/install.ps1 | iex
```
- To ensure Ollama application is running in the background, just open a browser and go to http://localhost:11434/ and it will display a message saying **"ollama is running"**. If this is not the case, then on the terminal give the command `ollama serve` and try again :)

<h1 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Ollama lets you download models from Ollama Hub or the Hugging Face Hub.</div></h1>

## c. What Models Can Ollama Run?

<h3 align="center"><div class="alert alert-success" style="margin: 20px">Search available models:  <a href="https://ollama.com/search">ollama.com/search</a>.</div></h3>

| Model Type | Capabilities | Representative families |
| --- | --- | --- |
| **Text-only LLMs** | Chat, reasoning, summarization, code | **[llama3.3](https://ollama.com/library/llama3.3)**, **[llama3.2](https://ollama.com/library/llama3.2)** (1b/3b — tiny and fast), **[gemma3](https://ollama.com/library/gemma3)**, **[qwen3](https://ollama.com/library/qwen3)**, **[mistral](https://ollama.com/library/mistral)**, **[phi4](https://ollama.com/library/phi4)** |
| **Reasoning Models** | Step-by-step reasoning, maths | **[deepseek-r1](https://ollama.com/library/deepseek-r1)** (1.5b–671b), **[gpt-oss](https://ollama.com/library/gpt-oss)** (20b/120b, OpenAI open-weight), **[qwq](https://ollama.com/library/qwq)**, **[magistral](https://ollama.com/library/magistral)** |
| **Code-Specific** | Programming, completion, debugging | **[qwen3-coder](https://ollama.com/library/qwen3-coder)**, **[qwen2.5-coder](https://ollama.com/library/qwen2.5-coder)**, **[deepseek-coder-v2](https://ollama.com/library/deepseek-coder-v2)**, **[codestral](https://ollama.com/library/codestral)**, **[starcoder2](https://ollama.com/library/starcoder2)** |
| **Multimodal (Vision)** | Image understanding + text | **[qwen3-vl](https://ollama.com/library/qwen3-vl)**, **[llama3.2-vision](https://ollama.com/library/llama3.2-vision)** (11b/90b), **[gemma3](https://ollama.com/library/gemma3)** (vision-capable sizes), **[llava](https://ollama.com/library/llava)**, **[minicpm-v](https://ollama.com/library/minicpm-v)** |
| **Small / Efficient** | Lightweight for laptops and edge | **[llama3.2:1b](https://ollama.com/library/llama3.2)**, **[gemma3:1b](https://ollama.com/library/gemma3)**, **[smollm2](https://ollama.com/library/smollm2)**, **[tinyllama](https://ollama.com/library/tinyllama)**, **[phi3](https://ollama.com/library/phi3)** |
| **Tool-calling / Agents** | Function calls, external APIs | Tool calling is a **capability of the model tag**, not a separate model. `llama3.1`, `llama3.3`, `qwen3`, `mistral-nemo`, `command-r-plus` and `hermes3` all support it — run `ollama show <model>` and look at the **capabilities** line. |
| **Embedding Models** | Semantic vectors for RAG | **[nomic-embed-text](https://ollama.com/library/nomic-embed-text)**, **[mxbai-embed-large](https://ollama.com/library/mxbai-embed-large)**, **[bge-m3](https://ollama.com/library/bge-m3)**, **[all-minilm](https://ollama.com/library/all-minilm)** |
| **MoE (Mixture of Experts)** | Large capacity, efficient inference | **[deepseek-v3](https://ollama.com/library/deepseek-v3)**, **[mixtral](https://ollama.com/library/mixtral)** (8x7b/8x22b), **[qwen3](https://ollama.com/library/qwen3)** MoE sizes, **[gpt-oss:120b](https://ollama.com/library/gpt-oss)** |

> **Reading a tag:** `llama3.2:1b` means family `llama3.2`, size `1b`. A `-cloud` suffix (e.g. `gpt-oss:20b-cloud`) means the weights run on Ollama's servers instead of your machine. No suffix means it runs locally and works offline.


# <span style='background :lightgreen' >3. Using Ollama through the GUI</span>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">We start with the graphical app, because it needs no commands at all — and we start with <b>cloud</b> models, because they work on any laptop regardless of how much RAM or GPU you have.</div></h3>

## a. Sign in (needed only for cloud models)
- Many new models are too large to fit on widely available GPUs, therefore, you can use **Ollama's cloud**, which is a way to run open models on datacenter-grade hardware while driving them from your local CLI, Python code, or any OpenAI-compatible client. Cloud models include hourly and daily limits to ensure fair usage.
- Cloud models require a free account on ollama.com. You can sign in **either** way:
    - **From the terminal (simplest):** `ollama signin` — this opens a browser to authenticate, then returns to the CLI. Sign out again with `ollama signout`.
    - **From the desktop app:** click the Ollama icon in the menu bar (macOS) or system tray (Windows) → **Sign In** → authorise in the browser.
- For scripts and CI where an interactive browser login is not possible, create an API key at https://ollama.com/settings/keys and export it:
```bash
export OLLAMA_API_KEY=your_key_here
```

## b. Free cloud-hosted models

Cloud models run on Ollama's datacenter hardware and stream the answer back to your machine — nothing is downloaded, and a 120-billion-parameter model works the same on a laptop as on a workstation. There are hourly and daily fair-use limits.

**Verified free on a standard ollama.com account (November 2026):**

| Model | Notes |
|---|---|
| `gpt-oss:20b-cloud` | OpenAI open-weight reasoning model — **the default used throughout this notebook** |
| `gpt-oss:120b-cloud` | Larger sibling, stronger reasoning |
| `gemma4:cloud` | Google Gemma 4 |
| `nemotron-3-super:cloud` | NVIDIA Nemotron 3 Super |

**Currently require a paid Ollama plan** (they will return *"this model requires a subscription"*): `qwen3.5`, `deepseek-v4-flash`, `glm-5.1`, `glm-5.2`, `kimi-k2.6`, `minimax-m2.7`, `kimi-k3`, `minimax-m3`.

<h3 align="center"><div class="alert alert-warning" style="margin: 20px">⚠️ <b>Ollama retires cloud models.</b> A retired tag returns <b>HTTP 410</b> naming the retirement date. <code>deepseek-v3.1:671b-cloud</code>, <code>qwen3-coder:480b-cloud</code>, <code>glm-5</code>, <code>kimi-k2.5</code>, <code>minimax-m2.5</code>, <code>qwen3-coder-next</code> and <code>qwen3-vl:235b</code> have all been retired. Check <a href="https://ollama.com/search?c=cloud">ollama.com/search?c=cloud</a> before relying on any cloud tag — and note that <b>a model appearing in the list does not mean your account can call it.</b></div></h3>

**Try it in the GUI:** open the Ollama app, pick `gpt-oss:20b-cloud` from the model selector, and ask it something. For cloud models the GUI also exposes toggles for **web search** and **thinking**.

## c. Downloading a small model to run on your own machine

Now the opposite case — a model that lives on your disk and runs on your CPU/GPU, with **no internet needed** after the download.

- In the Ollama app, search for **`llama3.2`** and pull the **1b** size (about 1.3 GB). Good small choices: `llama3.2:1b`, `gemma3:1b`, `tinyllama`.
- Once downloaded it appears in the model selector next to the cloud models. **Same interface, completely different execution**: your hardware, your electricity, your privacy — and no usage limits.
- Models from Ollama's library are **GGUF quantized** builds. GGUF ("GPT-Generated Unified Format") is a single-file binary format holding weights plus metadata, designed for fast local inference.

> **The mental model to carry forward:** a tag ending in `-cloud` runs on Ollama's servers; a tag without it runs on yours. Everything else — the GUI, the CLI, the Python code — is identical.

# <span style='background :lightgreen' >4. Using Ollama through the CLI</span>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">Everything you just did in the GUI, now from a terminal — this is what you will actually use day to day, and what scripts and agent harnesses drive underneath.</div></h3>

## a. Managing models
```bash
$ ollama                     # list the available commands
$ ollama signin              # sign in (needed once, for cloud models); `ollama signout` to reverse
$ ollama list                # every model you have pulled, local and cloud
$ ollama pull llama3.2:1b    # download a local model (stored under ~/.ollama/models/)
$ ollama show llama3.2:1b    # architecture, parameters, context window, quantization, capabilities, license
$ ollama ps                  # which models are currently loaded in memory
$ ollama stop llama3.2:1b    # unload a model to free RAM
$ ollama rm  tinyllama       # delete a model from disk
```

> `ollama show <model>` is the quickest way to check whether a model supports **tools** or **vision** — look at the `capabilities` line.

## b. Chatting with different local/cloud hosted Models
```bash
$ ollama run llama3.2:1b                       # interactive session; leave with /bye, /exit or /quit
$ ollama run llama3.2:1b "What is the capital of Pakistan?"               # one-shot prompt
$ ollama run llama3.2:1b --verbose "Explain gravity"                      # also print token counts and timings
$ ollama run gpt-oss:20b-cloud "Explain MoE models in 2 lines"            # a cloud model, same command
$ ollama run gemma4:cloud "Compare llama.cpp and ollama"                  # a cloud model, same command
$ ollama run nemotron-3-super:cloud "Compare open-source vs open-weight models" # a cloud model, same command
```



## c. Feeding files and web pages into a prompt
```bash
# I/O redirection - send a file's contents along with the question
$ ollama run llama3.2:1b "Extract the names of cricket players from this text" < ../data/names.txt

# the pipe operator does the same thing
$ cat ../data/names.txt | ollama run llama3.2:1b "Extract the names of cricket players from this text"

# scrape a web page and ask a question about it
$ curl https://www.arifbutt.me/ | ollama run llama3.2:1b "Give a one-line description of Dr. Arif Butt from this HTML"
```

In [1]:
! ollama run llama3.2:1b "Extract the names of cricket players from this text" < ../data/names.txt

]11;?\⠋ ⠹ ⠸ ⠼ ⠴ ⠦ ⠦ ⠇ ⠏ Here are the names of cricket players extracted from the text:

1. Imran Khan
2. Wasim Akram
3. Shahid Afridi
4. Babar Azam
5. Shaheen Shah Afridi
6. Shadab Khan



## d. Talking to the REST API directly with `curl`
Ollama's server exposes plain HTTP endpoints. This is worth seeing once, because every library later in this notebook is just a wrapper around these calls.

```bash
# /api/generate - single-shot text generation. STATELESS: it keeps no history.
$ curl -X POST http://localhost:11434/api/generate -d '{"model": "llama3.2:1b", "prompt": "What is the capital of Pakistan?", "stream": false}'

# /api/chat - conversational. You send the whole message list each time; roles are system/user/assistant.
$ curl -X POST http://localhost:11434/api/chat -d '{"model": "llama3.2:1b", "messages": [{"role": "user", "content": "Hi, my name is Arif."}], "stream": false}'

# to give the model memory, resend the earlier turns along with the new question
$ curl -X POST http://localhost:11434/api/chat -d '{"model": "llama3.2:1b", "messages": [{"role": "user", "content": "Hi, my name is Arif."}, {"role": "assistant", "content": "Nice to meet you, Arif!"}, {"role": "user", "content": "What is my name?"}], "stream": false}'
```

> **Note the pattern:** the server is stateless. "Memory" in a chat application is simply the client resending the conversation each time — exactly what you saw with the OpenAI APIs in earlier lectures.

In [2]:
! curl -X POST http://localhost:11434/api/generate -d '{"model": "llama3.2:1b", "prompt": "What is the capital of Pakistan?", "stream": false}'

{"model":"llama3.2:1b","created_at":"2026-09-18T04:50:07.000844Z","response":"The capital of Pakistan is Islamabad.","done":true,"done_reason":"stop","context":[128006,9125,128007,271,38766,1303,33025,2696,25,6790,220,2366,18,271,128009,128006,882,128007,271,3923,374,279,6864,315,17076,30,128009,128006,78191,128007,271,791,6864,315,17076,374,92532,13],"total_duration":118046083,"load_duration":5870416,"prompt_eval_count":32,"prompt_eval_cached_count":20,"prompt_eval_duration":46962000,"eval_count":8,"eval_duration":62465000}

## e. Available Commands:
```
$ ollama serve        # Start the Ollama server (listens on 127.0.0.1:11434)
$ ollama create       # Create a custom model from a Modelfile
$ ollama show         # Show a model's details: parameters, template, license
$ ollama run          # Run a model interactively (pulls it first if not local)
$ ollama stop         # Unload a running model from memory
$ ollama pull         # Download a model from the registry
$ ollama push         # Upload a model to the registry (requires sign-in)
$ ollama signin       # Sign in to ollama.com
$ ollama signout      # Sign out from ollama.com
$ ollama list         # List all models downloaded to disk
$ ollama ps           # List models currently loaded in memory
$ ollama cp           # Copy a model under a new name
$ ollama rm           # Delete a model from disk
$ ollama launch       # Launch an integration with Ollama
$ ollama help         # Show help for any command
```

In [1]:
!ollama list

]11;?\NAME                        ID              SIZE      MODIFIED      
mygemma:latest              92c207c73455    1.1 GB    2 hours ago      
qwen3-coder-next:cloud      aa626c11ae8d    -         3 months ago     
deepseek-v4-flash:cloud     ea027821675c    -         3 months ago     
kimi-k2.5:cloud             6d1c3246c608    -         3 months ago     
glm-5:cloud                 c313cd065935    -         3 months ago     
tinyllama-ft-peft:latest    c910d9ca7c69    1.2 GB    4 months ago     
qwen3.5:cloud               a7bf6f7891c3    -         5 months ago     
qwen3-vl:235b-cloud         86b3322ec200    -         5 months ago     
qwen3.5:397b-cloud          a7bf6f7891c3    -         5 months ago     
minimax-m2.5:cloud          c0d5751c800f    -         6 months ago     
deepseek-v3.1:671b-cloud    d3749919e45f    -         8 months ago     
gpt-oss:20b-cloud           875e8e3a629a    -         8 months ago     
gpt-oss:120b-cloud          569662207105    -         8 mon

In [2]:
!ollama show llama3.2:1b

]11;?\  Model
    architecture        llama     
    parameters          1.2B      
    context length      131072    
    embedding length    2048      
    quantization        Q8_0      

  Capabilities
    completion    
    tools         

  License
    LLAMA 3.2 COMMUNITY LICENSE AGREEMENT                 
    Llama 3.2 Version Release Date: September 25, 2024    
    ...                                                   



In [3]:
# A local model - runs entirely on this machine, no internet required
!ollama run llama3.2:1b "What is the capital of Pakistan? Answer in one line."

]11;?\⠙ ⠹ ⠸ The capital of Pakistan is Islamabad.



In [4]:
# A free cloud-hosted model - same command, runs on Ollama's hardware
!ollama run gpt-oss:20b-cloud "What is the capital of Pakistan? Answer in one line."

]11;?\⠙ ⠹ ⠸ ⠸ ⠴ ⠴ ⠧ Thinking...
User asks: "What is the capital of Pakistan? Answer in one line." Straight 
answer: Islamabad. Provide one line.
...done thinking.

Islamabad.



In [5]:
# Feeding a file into the prompt with I/O redirection
!ollama run llama3.2:1b "Extract the names of the cricket players from this text" < ../data/names.txt

]11;?\⠋ ⠹ Here are the names of the cricket players mentioned in the text:

1. Imran Khan
2. Wasim Akram
3. Shahid Afridi
4. Babar Azam
5. Shaheen Shah Afridi
6. Shadab Khan



# <span style='background :lightgreen' >5. Agent Harnesses on Ollama</span>

## a. What is a harness?
- A **harness** is the scaffolding *around* a model: the loop that reads your files, calls tools, runs shell commands, and feeds results back so the model can take the next step. The runtimes above (Ollama, llama.cpp, vLLM) *serve* a model; a harness *drives* one.
- Because most harnesses speak the OpenAI-compatible protocol, several can be pointed at a local Ollama server instead of a cloud provider — which is what makes local models genuinely useful for day-to-day work.

### The harnesses Ollama can launch

`ollama launch <name>` starts any of these and points it at your Ollama server. The **Launch name** column is exactly what you type.

| Harness | Launch name | Made by | Source | Notes |
|---|---|---|---|---|
| **[OpenCode](https://github.com/anomalyco/opencode)** | `opencode` | Anomaly (makers of SST) | Open | Provider-agnostic; the most popular open harness — **used as the main example in this course** |
| **[Claude Code](https://claude.com/claude-code)** | `claude` | Anthropic | Closed | Normally drives Claude models; `ollama launch` repoints it at Ollama |
| **[Codex](https://github.com/openai/codex)** | `codex` | OpenAI | Open | Tuned for OpenAI models; sandboxed execution |
| **[Pi](https://github.com/earendil-works/pi)** | `pi` | Earendil (Mario Zechner) | Open | Minimal terminal agent with four tools: read, write, edit, bash |
| **Hermes Agent** | `hermes` | Nous Research | Open | Also `hermes-desktop` for the GUI version |
| **[Cline](https://github.com/cline/cline)** | `cline` | Cline | Open | Popular VS Code agent |
| **[Qwen Code](https://github.com/QwenLM/qwen-code)** | `qwen` | Alibaba | Open | Tuned for Qwen models |
| **Kimi Code CLI** | `kimi` | Moonshot AI | Open | Tuned for Kimi models |
| **DeepSeek Harness** | `dsh` | DeepSeek | Open | Tuned for DeepSeek models |
| **Copilot CLI** | `copilot` | GitHub | Closed | GitHub Copilot in the terminal |
| **ChatGPT desktop** | `chatgpt` | OpenAI | Closed | Aliases: `codex-app`, `codex-desktop` |
| **[VS Code](https://code.visualstudio.com/)** | `vscode` | Microsoft | Open | Alias: `code` |
| **Droid** | `droid` | Factory | Closed | Agentic software development |
| **Muse Code** | `muse` | — | — | Alias: `muse-code` |
| **OpenClaw** | `openclaw` | — | Open | Aliases: `clawdbot`, `moltbot` |
| **OMP** | `omp` | — | — | |
| **Pool** | `pool` | — | — | |

> Run `ollama launch --help` on your own machine to confirm the list — integrations are added and removed between Ollama releases.

**Harnesses not in `ollama launch`** can still be pointed at Ollama by hand, because they accept an OpenAI-compatible endpoint — see section **d** below. Two worth knowing: **[Aider](https://aider.chat)** (git-native editing, `pip`-installable) and **[Goose](https://block.github.io/goose/)** (Block; extension-based).

- **Why this matters for this course:** a coding-capable local model (for example a `qwen3-coder` tag) served by Ollama and driven by OpenCode or Aider gives you a fully offline coding assistant with no per-token cost. Quality is below the frontier cloud models, but for routine edits and refactors it is often sufficient.
- **The landscape changes quickly** — harnesses appear, get acquired, and are retired. Check a project's own repository before adopting it rather than relying on any printed list, including this one.

## b. `ollama launch` — run any harness against Ollama

Ollama ships a **`launch`** command that configures a coding harness to talk to your Ollama server and then starts it. You do not have to set base URLs, export API keys, or edit any config file — Ollama wires it up for you.

```bash
$ ollama launch --help          # see every integration your version supports
$ ollama launch                 # no argument -> interactive menu, pick from a list
$ ollama launch opencode        # launch a specific harness
$ ollama launch opencode --model gpt-oss:120b-cloud     # ...with a specific model
$ ollama launch claude   --model gpt-oss:20b-cloud
$ ollama launch codex    --model nemotron-3-super:cloud
$ ollama launch hermes   --model gemma4:cloud
$ ollama launch pi       --model nemotron-3-super:cloud

$ ollama launch opencode --config                       # configure only, do not start
```
>- Several models Ollama lists as "cloud" are not free like `qwen3.5`, `deepseek-v4-flash`, `glm-5.1`, `glm-5.2`, `kimi-k2.6`, `minimax-m2.7`, `minimax-m3`



Useful flags: `--model <model>` picks the model, `--config` configures without launching, `--restore` puts an integration back to its default profile, `-y` auto-answers prompts, and anything after `--` is passed straight through to the harness (e.g. `ollama launch codex -- --sandbox workspace-write`).

> **Why this matters:** the same harness can be pointed at a frontier cloud API *or* at a model on your own laptop. `ollama launch` makes switching between those a one-word change, which is what makes local models practical for everyday coding work.

## c. Which models can you actually use with a harness?

A coding harness does not just chat — it **calls tools** (read file, write file, run command) in a loop. So the model must support **tool calling**. Check any model with:

```bash
$ ollama show <model>      # look at the "Capabilities" line for `tools`
```

### Free cloud-hosted models — best choice for agentic coding

```bash
$ ollama launch opencode --model gpt-oss:120b-cloud      # completion, tools, thinking
$ ollama launch opencode --model gpt-oss:20b-cloud       # faster, lighter
$ ollama launch opencode --model nemotron-3-super:cloud  # completion, tools, thinking
$ ollama launch claude   --model gpt-oss:120b-cloud      # same models, different harness
$ ollama launch codex    --model gemma4:cloud            # completion, tools, thinking, vision
$ ollama launch opencode --model llama3.2:latest         # fully offline, modest capability
```



- **Typical tools a coding harness gives the model:**
    - Read: Read the contents of a file
    - Write: Create a new file or overwrite one
    - Edit: Change part of an existing file
    - Bash: Run a shell command
    - WebFetch: Download a web page from a URL
    - WebSearch: Search the web
    - NotebookEdit: Edit a cell of a Jupyter notebook
    - MCP: Call a tool provided by an MCP server

# <span style='background :lightgreen' >6. Accessing Ollama Models Programmatically</span>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">Three ways to call the same local server — Ollama's own Python library, and then two OpenAI-compatible APIs. The second and third are why everything you learned in Lectures 1-7 transfers to local models unchanged.</div></h3>

## a. Using Ollama's Native Python Library
- Install with `uv add ollama`, then `ollama.chat()` is the main entry point for conversational calls.
- This library speaks Ollama's own protocol (`/api/chat`) and exposes Ollama-specific options directly.

In [3]:
import ollama
import rich

response = ollama.chat(
                        model = 'llama3.2:1b', # "gpt-oss:20b-cloud", "gpt-oss:120b-cloud", "gemma4:cloud", "nemotron-3-super:cloud"  <- all free
                        messages = [{'role': 'system',  'content': "You are a helpful assistant"}, {'role': 'user', 'content': "What is capital of Pakistan?"}],
                        stream = False,                # Default: False - set True for streaming
                        format=None,                   # Default: None - 'json' for JSON output
                        tools=None,                    # Default: None - List of tool definitions
                        options={                      # Model-specific options
                                'temperature': 0.7,         # Default: 0.8 (0.0-2.0, creativity)
                                'top_p': 0.9,               # Default: 0.9 (nucleus sampling)
                                }
                    )

print("Content:", response.message.content)
#rich.print(response)

Content: The capital of Pakistan is Islamabad.


## b. Using OpenAI's `Chat Completions` API
- Ollama also serves an **OpenAI-compatible** endpoint at `http://localhost:11434/v1`.
- That means you can use the ordinary `openai` package and simply repoint `base_url` — the rest of your code is unchanged from the cloud-provider lectures.
- Ollama does not check the API key, but the OpenAI client requires *some* string, so pass any placeholder.

In [1]:
from openai import OpenAI

# Create an OpenAI client instance and specify the base_url as 'http://localhost:11434/v1' → Instead of OpenAI’s cloud (https://api.openai.com/v1), it’s pointing to a local server.
# This tells the client: “Send requests to my local Ollama API server, not OpenAI’s servers.”
client = OpenAI(base_url="http://localhost:11434/v1", api_key='abc')   # Since Ollama don’t really need authentication. Give any string and it will work

# Use OpenAI's Chat Completions API (served by your local Ollama server)
response = client.chat.completions.create(
                                            model = "llama3.2:1b",   # a small LOCAL model; try "gpt-oss:20b-cloud" for a free cloud model. # "gpt-oss:20b-cloud", "gpt-oss:120b-cloud", "gemma4:cloud", "nemotron-3-super:cloud"  <- all free
                                            messages=[{'role': 'system',  'content': "You are a helpful assistant"}, {'role': 'user', 'content': "What is capital of Pakistan?"}],
                                            temperature=0.7,
                                            max_tokens=500,
                                            stream=False
                                        )
# Extract and print the output text
print(response.choices[0].message.content)

The capital of Pakistan is Islamabad.


## c. Using OpenAI's `Responses` API
- The newer Responses API also works against Ollama's compatible endpoint.
- Note the different response shape: `response.output_text` instead of `response.choices[0].message.content`.

In [2]:
from openai import OpenAI

# Create an OpenAI client instance and specify the base_url as 'http://localhost:11434/v1' → Instead of OpenAI’s cloud (https://api.openai.com/v1), it’s pointing to a local server.
# This tells the client: “Send requests to my local Ollama API server, not OpenAI’s servers.”
client = OpenAI(base_url="http://localhost:11434/v1", api_key="abc") # Since Ollama is running locally, so it don’t really need authentication. Give any string and it will work

# Using the Responses API
response = client.responses.create(
                                    model="gpt-oss:20b-cloud",      # "gpt-oss:20b-cloud", "gpt-oss:120b-cloud", "gemma4:cloud", "nemotron-3-super:cloud"  <- all free
                                    #input= "What is the color of the sky? Tell me in a single line."      
                                    input=[{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "What is the capital of Pakistan."}]
                                    )

# Extract and print the output text
print(response.output_text) 

The capital of Pakistan is **Islamabad**.


# <span style='background :lightgreen' >7. Hello-World Examples</span>
## Writing a helper function for our ease

The helper below defaults to `gpt-oss:20b-cloud` (free, no download). Pass `model="llama3.2:1b"` to any call to run the same example completely offline instead.

In [1]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

def ask_ollama(
                user_prompt: str,
                system_prompt: str = "You are a helpful assistant that provides concise answers.",
                model: str = "gpt-oss:20b-cloud",   # a LIVE cloud model; use "llama3.2:1b" to stay fully offline
                temperature: float = 0.7,
                stream: bool = False
            ):
    
    response = client.chat.completions.create(
    messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
    model=model,
    temperature=temperature,
    stream=stream
    )    
    
    if stream:                    # Return streaming generator if requested
        return response
    return response.choices[0].message.content   # Return the aggregated text output

## Examples (Question Answering)

In [5]:
user_prompt = "Which is the capital of Pakistan?"
response = ask_ollama(user_prompt=user_prompt, model="llama3.2:1b")
print(response)

The capital of Pakistan is Islamabad.


In [6]:
user_prompt = "Tell me a bedtime story of Ali baba chalees chor"

# Get streaming generator
response = ask_ollama(user_prompt=user_prompt, stream=True)

# Read streaming chunks
for chunk in response:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

**Ali Baba and the Gentle Forty Thieves**

Once upon a quiet desert night, a humble wood‑cutter named Ali Baba wandered near a hidden valley. In the distance he heard a strange sound—deep, rolling thunder, as if the earth itself was speaking.

When he drew closer, he discovered a narrow cave, its mouth yawning like a secret mouth. Curious, he stepped inside and found a glittering treasure: gold coins, jeweled crowns, and silver goblets. But the most astonishing thing was the cave’s tiny door, carved with an ancient word: **“Open Sesame.”** When he whispered the word, the door swung open, and the treasure poured into the cavern like a waterfall of light.

Ali Baba was amazed, but he also felt a pang of worry. He could not keep such a treasure without sharing it. He left the cave with a handful of coins, hoping to use them to help his family.

That night, a terrible storm blew across the desert. The wind howled, and the sand rose in swirling eddies. Ali Baba’s house was struck by a fierc

##  Question Answering from Content Passed

In [12]:
!cat ../data/names.txt

Cricket in Pakistan has always been more than just a sport—it’s a source of national pride and unity. Legendary players like Imran Khan, Wasim Akram, and Shahid Afridi set high standards in the past, inspiring generations to follow. Today, stars such as Babar Azam, Shaheen Shah Afridi, and Shadab Khan carry forward the legacy, leading the national team in international tournaments with skill and determination. Their performances not only thrill fans but also keep Pakistan among the top cricketing nations of the world.

Politics in Pakistan, meanwhile, remains dynamic and often turbulent, with key figures shaping the country’s direction. Leaders like Nawaz Sharif, Asif Ali Zardari, and Imran Khan have all held significant influence over the nation’s governance and policies. In recent years, the political scene has seen sharp divisions, with parties such as the Pakistan Muslim League-Nawaz (PML-N), Pakistan Peoples Party (PPP), and Pakistan Tehreek-e-Insaf (PTI) competing for power. Deba

In [2]:
from openai import OpenAI

with open("../data/names.txt", "r") as f:
    file_content = f.read()

user_prompt = f"Extract names from this text:\n{file_content}"
response = ask_ollama(user_prompt=user_prompt, model="llama3.2:1b")
print(response)

Here are the names mentioned in the text:

1. Imran Khan
2. Wasim Akram
3. Shahid Afridi
4. Babar Azam
5. Shaheen Shah Afridi
6. Shadab Khan
7. Nawaz Sharif
8. Asif Ali Zardari
9. Imran Khan (again, as he is mentioned multiple times)


## Examples (Binary Classification: Sentiment analysis, Spam detection, Medical diagnosis)

In [14]:
system_prompt = "You are an expert who will classify a sentense as having either a Positive or Negative sentiment."
user_prompt = "I love the youtube videos of Arif, as they are very informative"
response = ask_ollama(user_prompt=user_prompt, system_prompt=system_prompt)
print(response)

Positive


## Examples (Multi-class Classification)

In [15]:
system_prompt = "Classify product reviews into these categories: 'Electronics', 'Clothing', 'Books', 'Home & Garden', 'Sports', or 'Food'. \
Respond with only the category."
user_prompt = "This novel has an incredible plot twist that kept me reading all night"
response = ask_ollama(user_prompt=user_prompt, system_prompt=system_prompt)
print(response)

Books


In [16]:
system_role = "Classify product reviews into these categories: 'Electronics', 'Clothing', 'Books', 'Home & Garden', 'Sports', or 'Food'. \
Respond with only the category."
user_prompt = "The wireless headphones have excellent sound quality and battery life"
response = ask_ollama(user_prompt, system_role)
print(response)

Electronics


## Examples (Text Generation)

In [3]:
system_prompt = "You are an expert of political science and history and have a deep understanding of policical situation of Pakistan."
user_prompt = "Write down a 50 words summary about the fairness of general elections held in Pakistan on February 08, 2024."
response = ask_ollama(user_prompt=user_prompt, system_prompt=system_prompt, temperature=1.0)
print(response)

The February 8, 2024 general elections in Pakistan were broadly regarded as competitive, yet significant concerns emerged regarding fairness. Critics cited vote‑count discrepancies, limited opposition access to media, and allegations of intimidation in certain provinces. Despite independent monitors affirming procedural integrity, persistent allegations call for reforms to ensure transparent, inclusive electoral outcomes.


## Examples (Code Generation)

In [18]:
system_prompt = "You are an expert of C programing in C language."
user_prompt = "Write down a C program that generates first ten numbers of fibonacci sequence."
response = ask_ollama(user_prompt=user_prompt, system_prompt=system_prompt, stream=True)


# Read streaming chunks
for chunk in response:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

Here’s a compact, self‑contained C program that prints the first ten numbers of the Fibonacci sequence (starting from 0).  
Feel free to copy‑paste it into your favorite editor, compile with `gcc` (or any standard C compiler), and run it.

```c
/*  fib10.c  -  print first ten Fibonacci numbers
 *  Author: <your name>
 *  Date:   <today’s date>
 *
 *  The first ten Fibonacci numbers are:
 *      0 1 1 2 3 5 8 13 21 34
 *
 *  Compile:
 *      gcc -std=c11 -Wall -O2 -o fib10 fib10.c
 *
 *  Run:
 *      ./fib10
 */

#include <stdio.h>

int main(void)
{
    const int N = 10;          /* how many numbers to generate    */
    long long a = 0;           /* F(0) */
    long long b = 1;           /* F(1) */
    int i;

    for (i = 0; i < N; ++i) {
        printf("%lld ", a);   /* print current Fibonacci number */

        /* compute next number in the sequence */
        long long next = a + b;
        a = b;
        b = next;
    }

    putchar('\n');            /* tidy up the output line    

## Examples (Text Translation)

In [19]:
user_prompt = """
Please act as an expert of English to Urdu translator by translating the given sentence from English into Urdu.
'The budget this year will have a very bad impact on the low salried people'
"""
response = ask_ollama(user_prompt=user_prompt)
print(response)

اس سال کا بجٹ کم تنخواہ والے لوگوں پر بہت برا اثر ڈالے گا۔


## Examples (Text Summarization)

In [20]:
system_prompt = "You are an expert of English language."

user_prompt = f'''
Summarize the text below in at most 20 words:
```The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.```
'''

response = ask_ollama(user_prompt=user_prompt, system_prompt=system_prompt, temperature=0.2)
print(response)

Hugging Face transformers library: versatile, powerful NLP tool enabling classification, NER, QA; popular, lowers entry barrier for data scientists.


## Examples (Named Entity Recognition)

In [21]:
system_prompt = """You are a  Named Entity Recognition specialist. Extract and classify entities from the given text into these categories only if they exist:
- name
- major
- university
- nationality
- grades
- club
Format your response as: 'Entity: [text] | Type: [category]' with each entity on a new line."""

user_prompt = '''
Zelaid Mujahid is a sophomore majoring in Data Science at University of the Punjab. \
He is Pakistani national and has a 3.5 GPA. Mujahid is an active member of the department's AI Club.\
He hopes to pursue a career in AI after graduating.
'''

response = ask_ollama(user_prompt=user_prompt, system_prompt=system_prompt)
print(response)

Entity: Zelaid Mujahid | Type: name
Entity: Data Science | Type: major
Entity: University of the Punjab | Type: university
Entity: Pakistani | Type: nationality
Entity: 3.5 GPA | Type: grades
Entity: AI Club | Type: club


# <span style='background :lightgreen' >8. GGUF Models and llama.cpp</span>

Section **A** builds a custom Ollama model from a raw GGUF file downloaded from Hugging Face; section **B** runs the same GGUF file directly with `llama.cpp`.

## A. Building a Custom Model from a GGUF File

<img align="right" width="800"  src="../images/gguf_1.jpg"  >

### Downloading a GGUF model from Hugging Face
### Step 1: Find a GGUF model on Hugging Face
- Ollama's model library ships GGUF files, because its engine (llama.cpp) reads GGUF for efficient inference. A GGUF file is also the simplest thing to import into Ollama yourself
- Visit the models tab of hugging face, https://huggingface.co/models and search for "gguf" and you will see different models by unsloth, Qwen, Phi, Mistral and  TheBloke
- **GPT-Generated Unified Format (GGUF)** is a modern, unified model file format designed for efficient local inference, especially for *LLaMA-family* of models using tools like **llama.cpp** and **Ollama**.
    - *GGUF File Container:* The outer wrapper that holds all model components in a single binary file, eliminating the need for multi-file setups and ensures consistency between model components.
    - *Metadata Layer:* This layer stores model configuration and descriptive information, including model architecture and dimensions, context length, special tokens (BOS, EOS, PAD) and inference-related parameters.
    - *Tokenization Layer:* This layer defines how raw text is converted into tokens.
    - *Quantization Layer:* This layer describes how weights are compressed using different quantization formats like Q4, and Q8. This enables running large models on limited hardware.
    - *Model Weights and Tensors:* This layer contains the actual neural network parameters stored as optimized tensors.


### Step 2: Download one of the GGUF model to your computer
- Click on ‘Files and Versions’ on the model page: https://huggingface.co/unsloth/gemma-3-1b-it-GGUF/tree/main and select the version
- A model’s raw size in memory is roughly: `Parameters × Bytes per parameter`. The bytes per parameter depends on quantization level:
    - FP16 (16-bit float): 2 bytes/parameter
    - INT8 quantization: 1 byte/parameter
    - INT4 quantization: 0.5 bytes/parameter
- So memory use scales with both parameter count and quantization method.
    - 1.5B model → ~3–3.5 GB RAM (unquantized FP16).
    - 7B model → ~14–16 GB RAM (unquantized FP16).
    - With quantization, the requirement drops by 2×–4×, which is why Ollama and llama.cpp can run 7B models comfortably on machines with 8–16 GB RAM.

- **Download via Browser**
- Let us download the 8-bit quantized version from this link: https://huggingface.co/unsloth/gemma-3-1b-it-GGUF/blob/main/gemma-3-1b-it-Q8_0.gguf
- Click the download button or clone the repository.
- **Download Using Command Line**
```bash
# the hf command comes with the huggingface_hub package (it replaced huggingface-cli)
# Windows
hf download TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF tinyllama-1.1b-chat-v1.0.Q8_0.gguf --local-dir C:\models\TinyLlama_safe

# Linux/macOS
hf download TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF tinyllama-1.1b-chat-v1.0.Q8_0.gguf --local-dir ~/models/TinyLlama_safe
```

- I have downloaded models inside the `../models/gemma/` directory.

In [7]:
!ls -lh ../models/gemma/

total 2088504
-rw-r--r--@ 1 arif  staff   1.0G 19 Dec  2025 gemma-3-1b-it-Q8_0.gguf
-rw-r--r--@ 1 arif  staff   114B 19 Dec  2025 Modelfile


### Step 3: Create a Modelfile
- A **Modelfile** in Ollama is a small text configuration file that tells Ollama how to use a specific model.
- Think of it as a bridge between the GGUF model file and Ollama’s chat interface, it defines which model to load and how the conversation should behave.
- **Why is it needed?** Ollama itself cannot automatically know:
  - Which GGUF model to load.
  - How to handle prompts, messages, and system instructions.
  - What generation parameters to use (temperature, top-p, etc.).
- The Modelfile encapsulates all this information so you can run your model with consistent behavior.
- It makes customizing models easier without modifying the GGUF file itself.
- The only required section in the Modelfile is the `FROM` section, which points to your model file you just downloaded (where the model weights live)
```
FROM ../models/gemma/gemma-3-1b-it-Q8_0.gguf

SYSTEM "You are a helpful assistant that answers briefly."

PARAMETER temperature 0.7
```

In [8]:
!cat ../models/gemma/Modelfile

FROM gemma-3-1b-it-Q8_0.gguf
SYSTEM "You are a helpful assistant that answers briefly."
PARAMETER temperature 0.7


### Step 4 — Build the model using ollama create
- Use the `ollama create` command which will do the following tasks:
    - Copy your local GGUF file into Ollama’s internal model store, verifying its SHA-256 hash.
    - Parsed the GGUF to detect model architecture (Gemma-3-Instruct) and created new “layers” that include your system prompt and parameters.
    - Wrote a new manifest and registered a new Ollama model name, so you can run it like any other built-in model.
- Once done the `ollama list` command will display the new model that you can run

In [24]:
!ollama create mygemma -f ../models/gemma/Modelfile

]11;?\gathering model components ⠙ gathering model components ⠹ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠦ gathering model components ⠦ gathering model components 
copying file sha256:616dfb049ad14288d971d96f5ca4953fdebbf1e3cd407ad159f3bfd47090201d 0% ⠋ gathering model components 
copying file sha256:616dfb049ad14288d971d96f5ca4953fdebbf1e3cd407ad159f3bfd47090201d 4% ⠙ gathering model components 
copying file sha256:616dfb049ad14288d971d96f5ca4953fdebbf1e3cd407ad159f3bfd47090201d 14% ⠹ gathering model components 
copying file sha256:616dfb049ad14288d971d96f5ca4953fdebbf1e3cd407ad159f3bfd47090201d 24% ⠸ gathering model components 
copying file sha256:616dfb049ad14288d971d96f5ca4953fdebbf1e3cd407ad159f3bfd47090201d 30% ⠼ gathering model components 
copying file sha256:616dfb049ad14288d971d96f5ca4953fdebbf1e3cd407ad159f3bfd47090201d 41% ⠴ gathering model components 
copying file sha256:616dfb049ad14288d971d96f5ca49

In [25]:
!ollama list

]11;?\NAME                        ID              SIZE      MODIFIED      
mygemma:latest              92c207c73455    1.1 GB    5 seconds ago    
qwen3-coder-next:cloud      aa626c11ae8d    -         3 months ago     
deepseek-v4-flash:cloud     ea027821675c    -         3 months ago     
kimi-k2.5:cloud             6d1c3246c608    -         3 months ago     
glm-5:cloud                 c313cd065935    -         3 months ago     
tinyllama-ft-peft:latest    c910d9ca7c69    1.2 GB    4 months ago     
qwen3.5:cloud               a7bf6f7891c3    -         5 months ago     
qwen3-vl:235b-cloud         86b3322ec200    -         5 months ago     
qwen3.5:397b-cloud          a7bf6f7891c3    -         5 months ago     
minimax-m2.5:cloud          c0d5751c800f    -         6 months ago     
deepseek-v3.1:671b-cloud    d3749919e45f    -         8 months ago     
gpt-oss:20b-cloud           875e8e3a629a    -         8 months ago     
gpt-oss:120b-cloud          569662207105    -         8 mon

### Step 5 - Run your new Model via CLI

In [12]:
!ollama run mygemma:latest "What is the capital of Pakistan"

]11;?\Islamabad.



## B. Using llama.cpp Directly

### a. Download and Install llama.cpp
- There are different ways to download and install **llama.cpp** on your machine: https://github.com/ggml-org/llama.cpp
    1. Install llama.cpp using `brew install llama.cpp` on Mac or (use `winget` for Windows or `nix` for Linux)
    2. Download pre-built binaries from the releases page
    3. Run with Docker - see our Docker documentation
    4. Clone the repository, and build using `cmake`
### b. Inference Using llama.cpp 
- You can manually download the raw GGUF file from Hugging Face and access them via command line interface of llama.cpp
- llama.cpp does not automatically download models from Hugging Face unless you use the -hf flag.
- Models must be in GGUF format, otherwise you may need conversion using the provided conversion utilities in the llama.cpp repo.
- Below are the commands that you practice at your time (Replace `llama-cli` with `llama-cli.exe` for Windows):
```bash
$ llama-cli --help        # Shows general usage, options, and flags available for the CLI.
$ llama-cli -hf ggml-org/gemma-3-1b-it-GGUF    # Automatically downloads the specified Hugging Face GGUF model and runs it locally.
$ llama-cli -m /Users/arif/Documents/genai-course/models/gemma/gemma-3-1b-it-Q8_0.gguf # Run Interactive Session with a Local Model
$ llama-cli -m /Users/arif/Documents/genai-course/models/gemma/gemma-3-1b-it-Q8_0.gguf -n 256 --temp 0.7 # Run With Additional Parameters
```
- You can start the llama-server and load the model. The embedded server often exposes a lightweight Web UI at `http://localhost:PORT` where you can interact with the model via browser.
```bash
$ llama-server -m /Users/arif/Documents/genai-course/models/gemma/gemma-3-1b-it-Q8_0.gguf  --port 8080 
$ llama-server -m /Users/arif/Documents/genai-course/models/gemma/gemma-3-1b-it-Q8_0.gguf  -c 8192 -np 4 --port 8080  # -c specifies context window size. -np 4 specifies how many users or requests the server can handle simultaneously.
```

# <span style='background :lightgreen' >9. Beyond llama.cpp</span>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">Ollama and llama.cpp serve one person on one machine. This section looks at two tools built for the two extremes, vLLM for many users on big GPUs and AirLLM for huge models on small machines, and then builds a small inference server from scratch.</div></h3>

## a. vLLM: Serving Many Users at Once
- **The problem:** Ollama serves one person well. A company chatbot receives hundreds of requests at the same time, and answering them one after another leaves most of the GPU idle.
- **Continuous batching:** vLLM runs many requests together in one batch, and it does not wait for the whole batch to finish. As soon as one request is done, a waiting request takes its place, so the GPU stays busy.
- **PagedAttention:** every request needs GPU memory for its KV cache, and requests differ in length. vLLM stores the cache in small fixed-size blocks, like pages in an operating system, so very little memory is wasted and more requests fit on the GPU at once ([paper](https://arxiv.org/pdf/2309.06180)).
- **The result:** in [Red Hat's benchmark](https://developers.redhat.com/articles/2025/08/08/ollama-vs-vllm-deep-dive-performance-benchmarking) on one A100 GPU, vLLM peaked at 793 tokens/s, against 41 tokens/s for Ollama in its default configuration.
- **Same client code:** vLLM serves an OpenAI-compatible API, by default on port 8000, so you call it exactly as you called Ollama in section 6:
```bash
$ pip install vllm                                    # on Google Colab (GPU runtime) or a Linux machine with an NVIDIA GPU
$ vllm serve TinyLlama/TinyLlama-1.1B-Chat-v1.0       # downloads the model from Hugging Face, serves it on http://localhost:8000/v1
```
```python
client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
response = client.chat.completions.create(model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                                          messages=[{"role": "user", "content": "What is the capital of Pakistan?"}])
```
> **We do not install vLLM in the course environment.** vLLM 0.33 needs its own PyTorch version (2.14) and ships Linux-only packages, so it clashes with our `uv` project. Use it on Colab or a Linux GPU machine. On an Apple Silicon Mac, the community plugin [vllm-metal](https://github.com/vllm-project/vllm-metal) installs vLLM into its own environment with Homebrew: `brew tap vllm-project/vllm-metal https://github.com/vllm-project/vllm-metal`, then `brew install vllm-project/vllm-metal/vllm-metal`.

## b. AirLLM: A Huge Model on a Small Machine
- **The problem:** a 70B model needs about 140 GB of memory in FP16, and even a 4-bit version needs about 35 GB. That is far more than a laptop has.
- **The idea:** a transformer is a stack of layers that run one after another. AirLLM first saves the model as one file per layer on disk. During inference it loads one layer, runs it, frees its memory, and loads the next. So you only need memory for **one layer** at a time: for a 70B Llama with 80 layers, that is under 2 GB.
- **The price:** every new token has to read **all** the layers from disk again, so generation takes seconds per token, even for a small model. It also needs free disk space for the per-layer copy of the model.
- **Where it runs:** on an NVIDIA GPU (`device="cuda:0"`), on a plain CPU (`device="cpu"`) under Windows or Linux, and on Apple Silicon Macs through MLX (on a Mac, only Llama- and Mistral-style models).
- **Install:** already in the course environment (`uv sync`); elsewhere, `pip install airllm`

The cell below runs TinyLlama (1.1B) through AirLLM. TinyLlama fits in memory anyway, so this shows *how* AirLLM works rather than the memory saving: on the first run it writes one file per layer, and the timing shows the cost of reading every layer for every token.

In [1]:
import contextlib, io, sys, time, warnings
warnings.filterwarnings("ignore")
from airllm import AutoModel

quiet = contextlib.redirect_stderr(io.StringIO())    # hides AirLLM's progress bars (one bar for every generated token)

with quiet:
    model = AutoModel.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                                      device="cpu",                                 # no GPU needed; use "cuda:0" on an NVIDIA GPU (a Mac always uses MLX)
                                      layer_shards_saving_path="../models/airllm")  # where the one-file-per-layer copy is written

messages = [{"role": "user", "content": "What is the capital of Pakistan? Answer in one line."}]
prompt = model.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

start = time.time()
with quiet:
    if sys.platform == "darwin":       # Mac: AirLLM runs on Apple's MLX and returns the text directly
        import mlx.core as mx
        ids = model.tokenizer(prompt, return_tensors="np")["input_ids"]
        answer = model.generate(mx.array(ids), max_new_tokens=20).split("</s>")[0]
    else:                              # Windows / Linux: PyTorch, returns token IDs like Hugging Face
        ids = model.tokenizer(prompt, return_tensors="pt")["input_ids"]
        output = model.generate(ids.to(model.device), max_new_tokens=20, use_cache=True, return_dict_in_generate=True)
        answer = model.tokenizer.decode(output.sequences[0][ids.shape[1]:], skip_special_tokens=True)

print(answer.strip())
print(f"Time: {time.time() - start:.1f} seconds")

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

found_layers:{'model.embed_tokens.': True, 'model.layers.10.': True, 'model.layers.11.': True, 'model.layers.12.': True, 'model.layers.13.': True, 'model.layers.14.': True, 'model.layers.15.': True, 'model.layers.16.': True, 'model.layers.17.': True, 'model.layers.18.': True, 'model.layers.19.': True, 'model.layers.20.': True, 'model.layers.21.': True, 'model.layers.0.': True, 'model.layers.1.': True, 'model.layers.2.': True, 'model.layers.3.': True, 'model.layers.4.': True, 'model.layers.5.': True, 'model.layers.6.': True, 'model.layers.7.': True, 'model.layers.8.': True, 'model.layers.9.': True, 'model.norm.': True, 'lm_head.': True}
saved layers already found in ../models/airllm/splitted_model
The capital of Pakistan is Islamabad.
Time: 64.5 seconds


## c. Which of These Can Run on Your Laptop?

| Your laptop | vLLM | AirLLM |
|---|---|---|
| **Windows, no NVIDIA GPU** | ❌ Not supported on Windows (only through WSL), and very slow on a CPU | ✅ `device="cpu"` |
| **Linux, no GPU** | ⚠️ CPU packages exist, but need a recent CPU (AVX2/AVX-512) and extra setup | ✅ `device="cpu"` |
| **Mac with Apple Silicon** | ⚠️ Only through *vllm-metal*, installed separately (macOS 15 or later) | ✅ Through MLX (Llama- and Mistral-style models) |
| **NVIDIA GPU, or Google Colab** | ✅ What vLLM is built for | ✅ `device="cuda:0"` |

> **In short:** AirLLM runs on any laptop, slowly. vLLM is a server for GPU machines; try it on Colab, or with vllm-metal on a Mac.

# <span style='background :lightgreen' >10. Build Your Own Inference Server: PyTorch + FastAPI + uvicorn</span>

Ollama, llama.cpp and vLLM all do two jobs: an **engine** runs the model, and a **server** lets other programs use the engine over HTTP. We now write both jobs ourselves, in two small files (about 85 lines of code, not counting comments), to see what these runtimes do for us:

```
 Your Python code (OpenAI SDK)
          │  HTTP:  POST /v1/chat/completions
          ▼
 server.py: a FastAPI app, run by uvicorn           ← part b
          │  engine.generate(messages)
          ▼
 engine.py: PyTorch + Transformers                  ← part a
          chat template → token IDs → generate() → text
          ▼
 CPU / Apple GPU (MPS) / NVIDIA GPU
```

Part c then stops the server and frees the memory.

> **No GPU needed:** Qwen2.5-0.5B-Instruct (0.5B) runs well on a CPU. `generate()` comes from Transformers (Lec-12, Lec-21); the KV cache that makes it fast is explained in Lec-20.

## a. The Inference Engine
`engine.py` holds one class, `InferenceEngine`. Creating an engine loads the model once. Its `generate(messages)` method then answers a chat in five steps: chat template → token IDs → `generate()` → keep only the new tokens → text.

In [2]:
import os
os.makedirs("local_server", exist_ok=True)   # a folder for the two files of this section

In [3]:
%%writefile local_server/engine.py
# engine.py - the ENGINE layer: load a model with PyTorch, and turn chat messages into an answer
import threading
import torch
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # a small (0.5B) chat model from Hugging Face


def pick_device():
    """Return the fastest device this machine has."""
    if torch.cuda.is_available():               # an NVIDIA GPU
        return "cuda"
    if torch.backends.mps.is_available():       # the GPU of an Apple Silicon Mac
        return "mps"
    return "cpu"                                # no GPU: use the CPU


class InferenceEngine:
    """Loads the model once, then answers chat messages with generate()."""

    def __init__(self, model_id=MODEL_ID):
        self.model_id = model_id
        self.device = pick_device()

        # Each weight is a number. On a GPU we store it in 16 bits (half the memory);
        # on a CPU we keep 32 bits, because most CPUs are slow with 16-bit numbers.
        if self.device == "cpu":
            dtype = torch.float32
        else:
            dtype = torch.float16

        # The tokenizer turns text into token IDs and back (Lec-15)
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)

        # Download the weights (only the first time), load them, and move them to the device
        model = AutoModelForCausalLM.from_pretrained(
            model_id,                               # which model to load
            dtype=dtype,                            # 16-bit or 32-bit numbers, chosen above
        )
        self.model = model.to(self.device)

        # A lock lets only one request use the model at a time (our engine has no batching)
        self.lock = threading.Lock()

    def generate(self, messages, max_tokens=256, temperature=0.7):
        """Take a list of chat messages and return the model's answer as text."""

        # Step 1: messages -> one prompt string, in the format the model was trained on.
        #         (If there is no system message, Qwen's template adds its own:
        #          "You are Qwen, created by Alibaba Cloud. You are a helpful assistant.")
        prompt = self.tokenizer.apply_chat_template(
            messages,                               # [{"role": "user", "content": "..."}, ...]
            tokenize=False,                         # give us text, not token IDs (yet)
            add_generation_prompt=True,             # end with the "assistant:" marker, so the model answers
        )

        # Step 2: prompt string -> token IDs (a PyTorch tensor), on the same device as the model
        inputs = self.tokenizer(prompt, return_tensors="pt")
        inputs = inputs.to(self.device)
        prompt_length = inputs["input_ids"].shape[1]    # how many tokens the prompt has

        # Step 3: temperature 0 means "always pick the most likely token" (no sampling)
        if temperature > 0:
            do_sample = True
        else:
            do_sample = False
            temperature = None                      # generate() does not want a temperature without sampling

        # Step 4: run the model, one new token at a time, until it stops or reaches max_tokens
        with self.lock:                             # wait here if another request is using the model
            with torch.no_grad():                   # inference only: no gradients, less memory
                output = self.model.generate(
                    input_ids=inputs["input_ids"],              # the prompt, as token IDs
                    attention_mask=inputs["attention_mask"],    # which tokens to look at (all of them)
                    max_new_tokens=max_tokens,                  # the longest answer we allow
                    do_sample=do_sample,                        # random sampling or always the top token
                    temperature=temperature,                    # how random the sampling is
                )

        # Step 5: the output holds prompt + answer; keep only the new tokens, and turn them into text
        all_tokens = output[0]                      # we sent one prompt, so we take the first (only) row
        new_tokens = all_tokens[prompt_length:]     # drop the prompt tokens
        answer = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
        answer = answer.strip()                     # remove spaces and newlines at both ends
        return answer

Overwriting local_server/engine.py


**Run the engine.** Import the class and create one engine. This loads Qwen2.5-0.5B-Instruct into this notebook's memory: about 1 GB on a GPU (16-bit), or 2 GB on a CPU (32-bit). The first run also downloads the model (about 1 GB) from Hugging Face.

In [4]:
import sys
sys.path.append("local_server")             # let Python find engine.py inside the local_server folder

from engine import InferenceEngine          # the class we just wrote

engine = InferenceEngine()                  # load the tokenizer and the model weights (only once)

print("Model :", engine.model_id)
print("Device:", engine.device)

Model : Qwen/Qwen2.5-0.5B-Instruct
Device: mps


**Ask it questions.** Type a question and press Enter. Type `quit` (or press Enter on an empty line) to stop.

In [5]:
print("Ask the engine a question. Type 'quit' to stop.")
print()

while True:
    question = input("You: ")                       # wait until the user types a question
    if question == "" or question == "quit":        # an empty line or 'quit' ends the loop
        break

    messages = [{"role": "user", "content": question}]   # the chat format from section 6
    answer = engine.generate(                       # runs the model right here, inside this notebook
        messages,                                   # the conversation to answer
        temperature=0,                              # 0 = always the most likely token: steadier, more accurate answers
    )
    print("Model:", answer)
    print()

print("Bye!")

Ask the engine a question. Type 'quit' to stop.

You: hello
Model: Hello! How can I assist you today?

You: What is the capital of Pakistan?
Model: The capital of Pakistan is Islamabad.

You: Write one sentence about the moon.
Model: The moon is a celestial body that orbits Earth and reflects light from the Sun, making it an important part of our planet's natural environment.

You: quit
Bye!


## b. Wrapping the Engine in a Server
`server.py` imports the same class and creates one engine when the server starts. It then offers the engine at one HTTP endpoint, `POST /v1/chat/completions`. The endpoint reads the same JSON as OpenAI's Chat Completions API and answers in the same JSON shape, so any OpenAI client can use our server.

In [6]:
%%writefile local_server/server.py
# server.py - the SERVING layer: put the engine behind an OpenAI-compatible HTTP endpoint
import time
import uuid
from fastapi import FastAPI
from pydantic import BaseModel
from engine import InferenceEngine
from engine import MODEL_ID

# Load the model ONCE, when the server starts. Every request then reuses it.
engine = InferenceEngine()

# The web application. uvicorn runs it and passes every HTTP request to it.
app = FastAPI()


# What a client must send us: the same JSON fields as OpenAI's Chat Completions API.
# FastAPI reads the JSON body and checks it against this class for us.
class ChatRequest(BaseModel):
    model: str = MODEL_ID              # which model; we only have one, so this is just echoed back
    messages: list[dict]               # the conversation: [{"role": "user", "content": "..."}, ...]
    max_tokens: int = 256              # the longest answer the client wants
    temperature: float = 0.7           # 0 = always the same answer, higher = more varied


# POST http://localhost:8000/v1/chat/completions  ->  this function runs
@app.post("/v1/chat/completions")
def chat_completions(request: ChatRequest):
    # 1. Ask the engine for an answer
    answer = engine.generate(
        request.messages,                       # the conversation from the client
        max_tokens=request.max_tokens,          # the client's length limit
        temperature=request.temperature,        # the client's randomness setting
    )

    # 2. Pack the answer in the same JSON shape that OpenAI returns,
    #    so that the OpenAI SDK (and any other OpenAI client) can read it
    message = {"role": "assistant", "content": answer}
    choice = {"index": 0, "message": message, "finish_reason": "stop"}
    response = {
        "id": "chatcmpl-" + uuid.uuid4().hex,   # a random, unique ID for this answer
        "object": "chat.completion",            # tells the client what kind of object this is
        "created": int(time.time()),            # when it was made (seconds since 1970)
        "model": request.model,                 # which model answered
        "choices": [choice],                    # OpenAI can return several answers; we return one
    }
    return response                             # FastAPI turns the dict into JSON

Overwriting local_server/server.py


**Start the server.** In a terminal you would type `cd local_server` and then `uvicorn server:app --port 8000`. Here we start the same command from Python, in the background, so that the notebook stays free to act as the client. The server is a separate process, and it loads its **own copy** of the model.

In [7]:
import subprocess
import sys
import time
import urllib.request

# The command we would type in a terminal:  uvicorn server:app --port 8000
command = [
    sys.executable, "-m", "uvicorn",       # run uvicorn with this notebook's Python
    "server:app",                          # in file server.py, use the variable called app
    "--port", "8000",                      # listen on http://localhost:8000
]

# Start the server as a separate process, in the background
server = subprocess.Popen(
    command,
    cwd="local_server",                    # run inside local_server/, so that "from engine import ..." works
    stdout=subprocess.DEVNULL,             # hide the server's log messages
    stderr=subprocess.DEVNULL,
)

# Loading the model takes a few seconds. Knock on the server's door once a second until it answers.
server_ready = False
for second in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/docs")   # FastAPI's built-in documentation page
        server_ready = True
        break
    except Exception:
        time.sleep(1)                      # not ready yet: wait one second and try again

if server_ready:
    print("Server is running on http://localhost:8000")
else:
    print("The server did not start. Run the uvicorn command in a terminal to see the error.")

Server is running on http://localhost:8000


**Ask it questions.** The same loop as in part a, but now every question travels over HTTP to the server. This is the code from section 6b; only `base_url` and `model` change. While the server runs, you can also open http://localhost:8000/docs in a browser to try the endpoint by hand.

In [8]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",   # our own server, instead of OpenAI's
    api_key="not-needed",                  # our server checks no keys, but the SDK needs some value
)

print("Ask the server a question. Type 'quit' to stop.")
print()

while True:
    question = input("You: ")                       # wait until the user types a question
    if question == "" or question == "quit":        # an empty line or 'quit' ends the loop
        break

    messages = [{"role": "user", "content": question}]

    # This sends POST http://localhost:8000/v1/chat/completions and waits for the JSON answer
    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-0.5B-Instruct",            # the only model our server has
        messages=messages,                              # the conversation to answer
        temperature=0,                                  # 0 = always the most likely token, higher = more varied
        max_tokens=256,                                 # the longest answer we want
    )

    first_choice = response.choices[0]              # our server always returns exactly one choice
    answer = first_choice.message.content           # the same place as with OpenAI or Ollama
    print("Model:", answer)
    print()

print("Bye!")

Ask the server a question. Type 'quit' to stop.

You: Assalam o Alaikum
Model: Wa Alaikum Assalam! How can I assist you today?

You: What does CPU stand for?
Model: CPU stands for "Central Processing Unit". It is the core component of a computer that performs arithmetic and logical operations on data to process information. The CPU is responsible for executing instructions stored in memory and performing calculations or other tasks required by the operating system and applications running on the computer.

You: quit
Bye!


## c. Stopping the Server and Freeing Memory
Two copies of the model are in memory now: one inside the server process (part b), and one inside this notebook (part a). Stopping the server frees its copy, because the operating system takes back all the memory of a process that ends. To free the notebook's copy, we delete the `engine` object.

In [9]:
import gc
import torch

# 1. Stop the server. When the process ends, the operating system frees all of its memory.
server.terminate()                     # ask uvicorn to shut down
server.wait()                          # wait until it has really stopped
print("Server stopped")


def gpu_memory_gb():
    """GPU memory held by this notebook, in GB. A CPU-only machine keeps the model in RAM, so it shows 0."""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1e9
    if torch.backends.mps.is_available():
        return torch.mps.current_allocated_memory() / 1e9
    return 0.0


# 2. Free the notebook's own copy of the model (the engine from part a)
print(f"GPU memory before: {gpu_memory_gb():.2f} GB")

del engine                             # delete the only name that points to the engine (model + tokenizer)
gc.collect()                           # let Python clean up the objects nobody points to any more

if torch.cuda.is_available():
    torch.cuda.empty_cache()           # hand the freed memory back to the NVIDIA GPU
elif torch.backends.mps.is_available():
    torch.mps.empty_cache()            # hand the freed memory back to the Apple GPU

print(f"GPU memory after : {gpu_memory_gb():.2f} GB")

Server stopped
GPU memory before: 0.99 GB
GPU memory after : 0.00 GB


### What we did not build

| Feature | Our server | Ollama / llama.cpp | vLLM |
|---|---|---|---|
| Many users at once | One at a time (the lock) | A few in parallel, if configured (`OLLAMA_NUM_PARALLEL`, `-np`) | Continuous batching |
| Memory | Full FP32 / FP16 weights | Quantized GGUF (4-8 bit) | Quantization + PagedAttention |
| Streaming the answer | No | Yes | Yes |
| Choosing a model | Hard-coded in `engine.py` | `ollama pull` / `llama-server -hf` | Any Hugging Face model by name |
| Other endpoints | Only `/v1/chat/completions` | Also `/v1/responses`, `/v1/models`, embeddings | Also `/v1/responses`, `/v1/models`, embeddings |

> **This is why runtimes exist:** our two small files work, but turning them into something many people can use every day is the work that Ollama, llama.cpp and vLLM have already done.